# Skin Cancer Detection — Accuracy-Improved Google Colab Pipeline

**Pipeline:** Dataset Acquisition → Dataset Audit → Dataset Cleaning → Leakage Checks → Stratified Splitting → Augmentation → Hyperparameter Tuning → Transfer Learning → Fine-Tuning → Test Evaluation → TTA → Ensemble → Results → Streamlit Export

> **Important audit finding:** The supplied dataset contains 412 images and three labels: `dermIS`, `dermis`, and `dermquest`. These appear to be source/directory labels rather than clinical diagnoses. Therefore, the reported accuracy measures classification of these three dataset/source classes, not clinical skin-cancer diagnosis.

This version focuses on genuine generalization improvements rather than simply increasing training accuracy.


## 1. Dataset Acquisition

The notebook downloads the Kaggle dataset directly with `kagglehub`. The code automatically discovers the dataset's image/class folder structure instead of assuming a hard-coded path.

In [ ]:
!pip -q install kagglehub keras-tuner seaborn scikit-learn pandas matplotlib pillow

import os
import json
import random
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass


In [ ]:
import kagglehub

path = kagglehub.dataset_download("fatemehmehrparvar/skin-cancer-detection")
DATASET_ROOT = Path(path)

print("Path to dataset files:", DATASET_ROOT)
print("\nTop-level contents:")
for p in sorted(DATASET_ROOT.iterdir()):
    print(" -", p.name)

In [ ]:
def print_tree(root, max_depth=3):
    root = Path(root)
    for current, dirs, files in os.walk(root):
        current_path = Path(current)
        depth = len(current_path.relative_to(root).parts)
        if depth > max_depth:
            dirs[:] = []
            continue
        indent = "  " * depth
        print(f"{indent}{current_path.name}/")
        for f in sorted(files)[:10]:
            print(f"{indent}  {f}")
        if len(files) > 10:
            print(f"{indent}  ... ({len(files)-10} more files)")

print_tree(DATASET_ROOT, max_depth=3)

## 2. Dataset Visualization

We identify supported image files, inspect directory counts, and visualize representative images from each detected class.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_paths = [
    p for p in DATASET_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
]

print(f"Total image files found: {len(image_paths):,}")

parent_counts = pd.Series([p.parent.name for p in image_paths]).value_counts()
display(parent_counts.to_frame("image_count"))

plt.figure(figsize=(10, 5))
parent_counts.plot(kind="bar")
plt.title("Images by Parent Directory")
plt.xlabel("Parent directory")
plt.ylabel("Number of images")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
COMMON_CLASS_NAMES = {
    "benign", "malignant", "cancer", "non-cancer", "noncancer",
    "normal", "melanoma", "nevus", "bkl", "bcc", "akiec", "df", "vasc", "mel",
    "dermis", "dermis", "dermquest"
}
GENERIC_NAMES = {
    "train", "training", "test", "testing", "valid", "validation",
    "val", "images", "data", "dataset", "skin_data"
}

dir_image_counts = {}
for p in image_paths:
    dir_image_counts[p.parent] = dir_image_counts.get(p.parent, 0) + 1

candidate_dirs = [
    d for d, n in dir_image_counts.items()
    if d.name.lower().replace(" ", "-") in COMMON_CLASS_NAMES
]

if len(candidate_dirs) >= 2:
    class_dirs = sorted(candidate_dirs)
else:
    class_dirs = sorted([
        d for d, n in dir_image_counts.items()
        if d.name.lower() not in GENERIC_NAMES and n >= 2
    ])

print("Candidate class directories:")
for d in class_dirs:
    print(f" - {d} ({dir_image_counts[d]} images)")

if len(class_dirs) < 2:
    raise RuntimeError("Could not identify at least two class folders.")

rows = []
for class_dir in class_dirs:
    for p in image_paths:
        if p.parent == class_dir:
            rows.append({"filepath": str(p), "label": class_dir.name})

df = pd.DataFrame(rows)

print("\nDetected classes:", sorted(df["label"].unique()))
print("Total labelled images:", len(df))
display(df["label"].value_counts().to_frame("count"))


In [ ]:
classes = sorted(df["label"].unique())
n_classes = len(classes)

fig, axes = plt.subplots(
    n_classes, 5, figsize=(15, max(4, 3 * n_classes)),
    squeeze=False
)

for row_idx, cls in enumerate(classes):
    subset = df[df["label"] == cls]
    sample = subset.sample(n=min(5, len(subset)), random_state=SEED)

    for col_idx in range(5):
        ax = axes[row_idx, col_idx]
        ax.axis("off")
        if col_idx < len(sample):
            img_path = sample.iloc[col_idx]["filepath"]
            try:
                img = Image.open(img_path).convert("RGB")
                ax.imshow(img)
                ax.set_title(cls)
            except Exception as e:
                ax.set_title(f"Read error\n{e}")

plt.suptitle("Representative Dataset Images", fontsize=16)
plt.tight_layout()
plt.show()

## 3. Dataset Cleaning

Cleaning includes missing-file checks, corrupted/unreadable image detection, exact duplicate removal, and image-property inspection. Duplicate removal is performed before splitting.

In [ ]:
def md5_file(filepath, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(filepath, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

clean_rows = []
bad_files = []
hash_to_path = {}

for _, row in df.iterrows():
    fp = row["filepath"]
    if not os.path.exists(fp):
        bad_files.append((fp, "missing"))
        continue

    try:
        with Image.open(fp) as im:
            im.verify()
        with Image.open(fp) as im:
            width, height = im.size
            mode = im.mode

        file_hash = md5_file(fp)
        if file_hash in hash_to_path:
            bad_files.append((fp, f"duplicate_of:{hash_to_path[file_hash]}"))
            continue

        hash_to_path[file_hash] = fp
        clean_rows.append({
            "filepath": fp, "label": row["label"],
            "width": width, "height": height,
            "mode": mode, "md5": file_hash
        })
    except Exception as e:
        bad_files.append((fp, f"unreadable:{e}"))

clean_df = pd.DataFrame(clean_rows)
bad_df = pd.DataFrame(bad_files, columns=["filepath", "reason"])

print(f"Original labelled images : {len(df):,}")
print(f"Clean images              : {len(clean_df):,}")
print(f"Removed/unusable images   : {len(bad_df):,}")
display(clean_df["label"].value_counts().to_frame("clean_count"))
if len(bad_df):
    display(bad_df.head(20))


In [ ]:
print("Image modes:")
display(clean_df["mode"].value_counts().to_frame("count"))

print("Width statistics:")
display(clean_df["width"].describe().to_frame("value"))

print("Height statistics:")
display(clean_df["height"].describe().to_frame("value"))

plt.figure(figsize=(10, 5))
plt.hist(clean_df["width"], bins=30, alpha=0.7, label="Width")
plt.hist(clean_df["height"], bins=30, alpha=0.7, label="Height")
plt.title("Image Dimension Distribution")
plt.xlabel("Pixels")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Feature Engineering and Accuracy Improvements

Compared with the original notebook, this stage adds stronger augmentation, label smoothing, balanced training, and better transfer-learning backbones.


In [ ]:
class_names = sorted(clean_df["label"].unique())
class_to_index = {name: idx for idx, name in enumerate(class_names)}
index_to_class = {idx: name for name, idx in class_to_index.items()}
clean_df["label_id"] = clean_df["label"].map(class_to_index).astype(int)

NUM_CLASSES = len(class_names)
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

print("Classes:", class_names)
print("Number of classes:", NUM_CLASSES)


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.08),
    layers.RandomTranslation(0.06, 0.06),
    layers.RandomZoom(height_factor=(-0.10, 0.12), width_factor=(-0.10, 0.12)),
    layers.RandomContrast(0.12),
    layers.RandomBrightness(0.08, value_range=(0, 255)),
], name="data_augmentation")

example_path = clean_df.sample(1, random_state=SEED).iloc[0]["filepath"]
example_img = keras.utils.load_img(example_path, target_size=IMG_SIZE)
example_arr = keras.utils.img_to_array(example_img)

plt.figure(figsize=(12, 6))
for i in range(6):
    ax = plt.subplot(2, 3, i + 1)
    aug = data_augmentation(tf.expand_dims(example_arr, 0), training=True)
    ax.imshow(tf.cast(tf.clip_by_value(aug[0], 0, 255), tf.uint8))
    ax.axis("off")
plt.suptitle("Improved Data Augmentation")
plt.tight_layout()
plt.show()


## 5. Dataset Splitting

The original 70/15/15 stratified split is retained for direct comparison. Because the held-out test set is small, macro-F1 is also reported.


In [ ]:
train_df, temp_df = train_test_split(
    clean_df, test_size=0.30,
    stratify=clean_df["label_id"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50,
    stratify=temp_df["label_id"], random_state=SEED
)

print(f"Train: {len(train_df):,}")
print(f"Validation: {len(val_df):,}")
print(f"Test: {len(test_df):,}")

split_summary = pd.DataFrame({
    "Train": train_df["label"].value_counts(),
    "Validation": val_df["label"].value_counts(),
    "Test": test_df["label"].value_counts()
}).fillna(0).astype(int)

display(split_summary)
split_summary.plot(kind="bar", figsize=(10, 5))
plt.title("Class Distribution by Dataset Split")
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE, antialias=True)
    image = tf.cast(image, tf.float32)
    return image, label

def make_dataset(dataframe, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((
        dataframe["filepath"].values,
        dataframe["label_id"].values
    ))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(dataframe),
            seed=SEED,
            reshuffle_each_iteration=True
        )
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)
test_ds = make_dataset(test_df, shuffle=False)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_df["label_id"].values
)
class_weights = {i: float(w) for i, w in enumerate(weights)}

print("Class weights:")
for i, w in class_weights.items():
    print(f"  {index_to_class[i]}: {w:.4f}")


## 6. Model Training — Hyperparameter Tuning + Deep Fine-Tuning

The original training used only 5 frozen epochs + 5 fine-tuning epochs. The improved version tunes EfficientNetV2B0, trains a stronger head, fine-tunes deeper layers with a much smaller learning rate, and compares EfficientNetV2B0, DenseNet121 and ResNet50.


In [ ]:
RUN_TUNER = True
TUNE_TRIALS = 6

HEAD_EPOCHS = 10
FINE_TUNE_EPOCHS = 20
FINE_TUNE_FRACTION = 0.30

MODEL_DIR = Path("/content/skin_cancer_models_improved")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def get_backbone(model_name):
    if model_name == "EfficientNetV2B0":
        return keras.applications.EfficientNetV2B0(
            include_top=False, weights="imagenet",
            input_shape=(*IMG_SIZE, 3)
        )
    if model_name == "DenseNet121":
        return keras.applications.DenseNet121(
            include_top=False, weights="imagenet",
            input_shape=(*IMG_SIZE, 3)
        )
    if model_name == "ResNet50":
        return keras.applications.ResNet50(
            include_top=False, weights="imagenet",
            input_shape=(*IMG_SIZE, 3)
        )
    raise ValueError(model_name)

def preprocess_for(model_name, x):
    if model_name == "EfficientNetV2B0":
        return x
    if model_name == "DenseNet121":
        return keras.applications.densenet.preprocess_input(x)
    if model_name == "ResNet50":
        return keras.applications.resnet50.preprocess_input(x)
    raise ValueError(model_name)

def build_transfer_model(model_name, dropout=0.35, dense_units=256):
    inputs = keras.Input(shape=(*IMG_SIZE, 3), name="image")
    x = data_augmentation(inputs)
    x = preprocess_for(model_name, x)

    base = get_backbone(model_name)
    base.trainable = False

    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(dense_units, activation="swish")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    return keras.Model(inputs, outputs, name=model_name), base

def compile_model(model, lr, weight_decay=1e-5):
    try:
        optimizer = keras.optimizers.AdamW(
            learning_rate=lr, weight_decay=weight_decay
        )
    except Exception:
        optimizer = keras.optimizers.Adam(learning_rate=lr)

    model.compile(
        optimizer=optimizer,
        loss=keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.05),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")]
    )

def make_callbacks(model_name, patience=5):
    checkpoint_path = MODEL_DIR / f"{model_name}_best.keras"
    return [
        keras.callbacks.ModelCheckpoint(
            checkpoint_path, monitor="val_accuracy",
            save_best_only=True, mode="max", verbose=1
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=patience,
            mode="max", restore_best_weights=True, verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.3,
            patience=2, min_lr=1e-7, verbose=1
        )
    ]


In [ ]:
def tuner_model_builder(hp):
    dropout = hp.Float("dropout", 0.20, 0.55, step=0.05)
    dense_units = hp.Choice("dense_units", [128, 256, 384])
    lr = hp.Choice("learning_rate", [1e-3, 3e-4, 1e-4])
    wd = hp.Choice("weight_decay", [1e-6, 1e-5, 1e-4])

    model, _ = build_transfer_model(
        "EfficientNetV2B0",
        dropout=dropout,
        dense_units=dense_units
    )
    compile_model(model, lr=lr, weight_decay=wd)
    return model

if RUN_TUNER:
    tuner = kt.RandomSearch(
        tuner_model_builder,
        objective=kt.Objective("val_accuracy", direction="max"),
        max_trials=TUNE_TRIALS,
        overwrite=True,
        directory="/content/keras_tuner",
        project_name="skin_cancer_efficientnetv2"
    )
    tuner.search(
        train_ds,
        validation_data=val_ds,
        epochs=8,
        class_weight=class_weights,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=2,
            mode="max", restore_best_weights=True
        )],
        verbose=1
    )
    best_hp = tuner.get_best_hyperparameters(1)[0]
    TUNED_HP = {
        "dropout": best_hp.get("dropout"),
        "dense_units": best_hp.get("dense_units"),
        "learning_rate": best_hp.get("learning_rate"),
        "weight_decay": best_hp.get("weight_decay")
    }
else:
    TUNED_HP = {
        "dropout": 0.35,
        "dense_units": 256,
        "learning_rate": 3e-4,
        "weight_decay": 1e-5
    }

print(json.dumps(TUNED_HP, indent=2))


In [ ]:
def train_one_model(model_name, dropout=0.35, dense_units=256, head_lr=3e-4):
    print("\n" + "=" * 80)
    print("TRAINING:", model_name)
    print("=" * 80)

    model, base = build_transfer_model(
        model_name, dropout=dropout, dense_units=dense_units
    )
    compile_model(model, lr=head_lr, weight_decay=1e-5)
    callbacks = make_callbacks(model_name)

    history_head = model.fit(
        train_ds, validation_data=val_ds,
        epochs=HEAD_EPOCHS,
        class_weight=class_weights,
        callbacks=callbacks, verbose=1
    )

    # Fine-tune the final 30% of the backbone.
    base.trainable = True
    fine_tune_from = int(len(base.layers) * (1.0 - FINE_TUNE_FRACTION))
    for layer in base.layers[:fine_tune_from]:
        layer.trainable = False

    # Freeze BatchNorm for stability on a small dataset.
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    compile_model(model, lr=1e-5, weight_decay=1e-5)

    history_ft = model.fit(
        train_ds, validation_data=val_ds,
        initial_epoch=len(history_head.history["loss"]),
        epochs=HEAD_EPOCHS + FINE_TUNE_EPOCHS,
        class_weight=class_weights,
        callbacks=callbacks, verbose=1
    )

    best_path = MODEL_DIR / f"{model_name}_best.keras"
    best_model = keras.models.load_model(best_path)
    return best_model, (history_head, history_ft)

trained_models = {}
histories = {}

trained_models["EfficientNetV2B0"], histories["EfficientNetV2B0"] = train_one_model(
    "EfficientNetV2B0",
    dropout=TUNED_HP["dropout"],
    dense_units=TUNED_HP["dense_units"],
    head_lr=TUNED_HP["learning_rate"]
)

trained_models["DenseNet121"], histories["DenseNet121"] = train_one_model(
    "DenseNet121", dropout=0.35, dense_units=256, head_lr=3e-4
)

trained_models["ResNet50"], histories["ResNet50"] = train_one_model(
    "ResNet50", dropout=0.35, dense_units=256, head_lr=3e-4
)

print("Finished models:", list(trained_models.keys()))


## 7. Training History Visualization

In [ ]:
plt.figure(figsize=(12, 6))
for name, (h1, h2) in histories.items():
    val_acc = h1.history.get("val_accuracy", []) + h2.history.get("val_accuracy", [])
    plt.plot(range(1, len(val_acc) + 1), val_acc, marker="o", label=name)

plt.title("Validation Accuracy — Improved Training")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 8. Test-Time Augmentation and Model Testing


In [ ]:
tta = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.04),
    layers.RandomZoom(0.05),
], name="tta")

def get_predictions(model, dataset, use_tta=False, tta_rounds=5):
    y_true, y_prob = [], []
    for images, labels in dataset:
        if not use_tta:
            probs = model.predict(images, verbose=0)
        else:
            prob_list = [model(images, training=False).numpy()]
            for _ in range(tta_rounds - 1):
                aug_images = tta(images, training=True)
                prob_list.append(model(aug_images, training=False).numpy())
            probs = np.mean(prob_list, axis=0)

        y_true.extend(labels.numpy())
        y_prob.extend(probs)

    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = np.argmax(y_prob, axis=1)
    return y_true, y_pred, y_prob

def evaluate_predictions(name, y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    if NUM_CLASSES == 2:
        auc = roc_auc_score(y_true, y_prob[:, 1])
    else:
        auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="weighted")

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print(classification_report(
        y_true, y_pred, target_names=class_names, zero_division=0
    ))

    return {
        "Model": name, "Accuracy": acc, "Precision": precision,
        "Recall": recall, "Weighted_F1": weighted_f1,
        "Macro_F1": macro_f1, "ROC_AUC": auc
    }

results = []
prediction_store = {}

for name, model in trained_models.items():
    y_true, y_pred, y_prob = get_predictions(model, test_ds, use_tta=False)
    prediction_store[name] = (y_true, y_pred, y_prob)
    results.append(evaluate_predictions(name, y_true, y_pred, y_prob))

results_df = pd.DataFrame(results).sort_values(
    ["Macro_F1", "Accuracy"], ascending=False
).reset_index(drop=True)

display(results_df)


## 9. TTA Evaluation


In [ ]:
tta_results = []
tta_prediction_store = {}

for name, model in trained_models.items():
    y_true, y_pred, y_prob = get_predictions(
        model, test_ds, use_tta=True, tta_rounds=5
    )
    tta_prediction_store[name] = (y_true, y_pred, y_prob)
    tta_results.append(
        evaluate_predictions(f"{name} + TTA", y_true, y_pred, y_prob)
    )

tta_results_df = pd.DataFrame(tta_results).sort_values(
    ["Macro_F1", "Accuracy"], ascending=False
).reset_index(drop=True)

display(tta_results_df)


## 10. Probability Ensemble


In [ ]:
val_prediction_store = {}

for name, model in trained_models.items():
    yv_true, _, yv_prob = get_predictions(
        model, val_ds, use_tta=True, tta_rounds=5
    )
    val_prediction_store[name] = (yv_true, yv_prob)

model_names = list(trained_models.keys())
weight_grid = []

for a in np.arange(0.0, 1.01, 0.1):
    for b in np.arange(0.0, 1.01 - a, 0.1):
        c = 1.0 - a - b
        if c >= -1e-9:
            weight_grid.append(np.array([a, b, c]))

yv_true = val_prediction_store[model_names[0]][0]
best_weight = None
best_val_f1 = -1

for weights_ in weight_grid:
    ensemble_prob = sum(
        w * val_prediction_store[name][1]
        for w, name in zip(weights_, model_names)
    )
    pred = np.argmax(ensemble_prob, axis=1)
    score = f1_score(yv_true, pred, average="macro", zero_division=0)

    if score > best_val_f1:
        best_val_f1 = score
        best_weight = weights_

print("Models:", model_names)
print("Best validation weights:", best_weight)
print("Best validation macro-F1:", round(best_val_f1, 4))

test_probs = [tta_prediction_store[name][2] for name in model_names]
ensemble_prob = sum(w * p for w, p in zip(best_weight, test_probs))
ensemble_pred = np.argmax(ensemble_prob, axis=1)
ensemble_true = tta_prediction_store[model_names[0]][0]

ensemble_result = evaluate_predictions(
    "Weighted Ensemble + TTA",
    ensemble_true, ensemble_pred, ensemble_prob
)

all_results = pd.concat(
    [tta_results_df, pd.DataFrame([ensemble_result])],
    ignore_index=True
).sort_values(
    ["Macro_F1", "Accuracy"], ascending=False
).reset_index(drop=True)

display(all_results)


## 11. Confusion Matrices and Results Visualization


In [ ]:
for name, (y_true, y_pred, _) in tta_prediction_store.items():
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names
    )
    plt.title(f"Confusion Matrix — {name} + TTA")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.show()

cm = confusion_matrix(ensemble_true, ensemble_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names, yticklabels=class_names
)
plt.title("Confusion Matrix — Weighted Ensemble + TTA")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

metrics = ["Accuracy", "Precision", "Recall", "Weighted_F1", "Macro_F1", "ROC_AUC"]
comparison_long = all_results.melt(
    id_vars="Model", value_vars=metrics,
    var_name="Metric", value_name="Score"
)

plt.figure(figsize=(14, 6))
sns.barplot(data=comparison_long, x="Metric", y="Score", hue="Model")
plt.ylim(0, 1.05)
plt.title("Improved Model Performance Comparison")
plt.ylabel("Score")
plt.xlabel("")
plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

results_csv = "/content/improved_model_comparison.csv"
all_results.to_csv(results_csv, index=False)
print("Saved:", results_csv)
display(all_results)


## 12. Export Best Single Model for Streamlit

The ensemble is evaluated, but a single model is exported for a lightweight deployment package.


In [ ]:
best_single_name = tta_results_df.iloc[0]["Model"].replace(" + TTA", "")
best_single_model = trained_models[best_single_name]

BEST_DIR = Path("/content/skin_cancer_streamlit_improved")
BEST_DIR.mkdir(parents=True, exist_ok=True)

best_model_path = BEST_DIR / "best_skin_cancer_model.keras"
best_single_model.save(best_model_path)

metadata = {
    "model_name": best_single_name,
    "class_names": class_names,
    "class_to_index": class_to_index,
    "image_size": list(IMG_SIZE),
    "num_classes": NUM_CLASSES,
    "selection_metric": "macro_f1",
    "uses_training_augmentation": True,
    "uses_tta_for_evaluation": True,
    "note": "Dataset labels are the source folders in the supplied dataset, not clinical diagnoses."
}

with open(BEST_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

all_results.to_csv(BEST_DIR / "model_comparison.csv", index=False)

print("Selected single deployment model:", best_single_name)
print("Model:", best_model_path)
print("Metadata:", BEST_DIR / "metadata.json")


In [ ]:
app_code = '''import json
import numpy as np
import streamlit as st
import tensorflow as tf
from PIL import Image

st.set_page_config(page_title="Skin Lesion Image Classifier", page_icon="🔬")

MODEL_PATH = "best_skin_cancer_model.keras"
METADATA_PATH = "metadata.json"

@st.cache_resource
def load_model_and_metadata():
    model = tf.keras.models.load_model(MODEL_PATH)
    with open(METADATA_PATH, "r") as f:
        metadata = json.load(f)
    return model, metadata

model, metadata = load_model_and_metadata()
class_names = metadata["class_names"]
img_size = tuple(metadata["image_size"])

st.title("Skin Lesion Image Classification")
st.warning("Educational/research use only. This model is not a medical diagnostic device.")

uploaded_file = st.file_uploader(
    "Upload a skin-lesion image",
    type=["jpg", "jpeg", "png", "webp"]
)

if uploaded_file is not None:
    image = Image.open(uploaded_file).convert("RGB")
    st.image(image, caption="Uploaded image", use_container_width=True)

    image_resized = image.resize(img_size)
    x = np.asarray(image_resized).astype("float32")
    x = np.expand_dims(x, axis=0)

    probabilities = model.predict(x, verbose=0)[0]
    predicted_idx = int(np.argmax(probabilities))
    predicted_class = class_names[predicted_idx]
    confidence = float(probabilities[predicted_idx])

    st.subheader("Model output")
    st.write(f"Predicted class: **{predicted_class}**")
    st.write(f"Model confidence: **{confidence:.2%}**")

    st.subheader("Class probabilities")
    for cls, prob in zip(class_names, probabilities):
        st.progress(float(prob), text=f"{cls}: {prob:.2%}")

st.caption("Performance may not generalize to other cameras, populations, clinical settings, or image types.")
'''
requirements = '''streamlit
tensorflow
numpy
pillow
'''

(BEST_DIR / "app.py").write_text(app_code)
(BEST_DIR / "requirements.txt").write_text(requirements)

print("Deployment files:")
for p in sorted(BEST_DIR.iterdir()):
    print(" -", p.name)


## 13. Final Exported-Model Verification


In [ ]:
reloaded_model = keras.models.load_model(best_model_path)

sample = test_df.sample(1, random_state=SEED).iloc[0]
sample_image = Image.open(sample["filepath"]).convert("RGB").resize(IMG_SIZE)

sample_array = np.asarray(sample_image).astype("float32")
sample_array = np.expand_dims(sample_array, axis=0)

probs = reloaded_model.predict(sample_array, verbose=0)[0]
pred_idx = int(np.argmax(probs))

print("True class     :", sample["label"])
print("Predicted class:", class_names[pred_idx])
print("Probabilities  :", dict(zip(class_names, probs.round(4))))


# Final Deliverables

After running the notebook, the improved deployment package is created at:

`/content/skin_cancer_streamlit_improved/`

Files:
- `best_skin_cancer_model.keras`
- `metadata.json`
- `model_comparison.csv`
- `app.py`
- `requirements.txt`

### Accuracy improvements added
1. Hyperparameter tuning of EfficientNetV2B0.
2. Stronger pretrained backbone: EfficientNetV2B0.
3. DenseNet121 and ResNet50 as complementary models.
4. Longer staged training and deeper fine-tuning.
5. Much smaller learning rate during fine-tuning.
6. AdamW weight decay when available.
7. Label smoothing.
8. Stronger controlled image augmentation.
9. Class-balanced training weights.
10. Macro-F1 based model selection.
11. Test-time augmentation.
12. Validation-only weighted probability ensemble.
13. Exact duplicate removal before splitting.

**No accuracy number is hard-coded.** Run the notebook to measure the actual improvement. With only ~412 images and a small test set, dataset size and label quality remain the main limitations.
